## Vector Database in the RAG Pipeline

A **Vector Database** stores the **embeddings (vectors)** of document chunks. Instead of searching by exact keywords, it searches by **meaning (semantic similarity)** to find the most relevant information.

### Why Do We Need It?

Suppose your company has thousands of documents.

A user asks:

> **Can interns work remotely?**

Instead of searching every document, the vector database quickly finds the most relevant chunk.

---

### RAG Pipeline

```text
Documents
    │
    ▼
Load Documents
    │
    ▼
Split into Chunks
    │
    ▼
Generate Embeddings
    │
    ▼
Store in Vector Database
    ▲
    │
User Question
    │
    ▼
Generate Query Embedding
    │
    ▼
Similarity Search
    │
    ▼
Retrieve Top Chunks
    │
    ▼
LLM
    │
    ▼
Final Answer
```

---

### Real-Life Example

Stored document:

```
Interns may work remotely
with manager approval.
```

User asks:

> **Can interns work from home?**

The vector database understands that **"work remotely"** and **"work from home"** have similar meanings and retrieves the correct document.

The LLM then uses that context to generate an accurate answer.

---

### Popular Vector Databases

- PostgreSQL + pgvector
- Pinecone
- Qdrant
- Chroma
- FAISS
- Weaviate

---

### Summary

- Stores **document embeddings**
- Performs **semantic search**
- Retrieves the most relevant document chunks
- Provides context to the LLM
- Helps reduce hallucinations in RAG applications


In [ ]:
import os
from langchain_community.document_loaders import (
    DirectoryLoader,
    PyMuPDFLoader,
    PyPDFLoader,
)
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from pathlib import Path
import hashlib
from langchain_google_genai import GoogleGenerativeAIEmbeddings

EMBEDDING_MODEL = os.getenv("EMBEDDING_MODEL")
GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")

In [ ]:
embeddings_model = GoogleGenerativeAIEmbeddings(
    model=EMBEDDING_MODEL,
    google_api_key=GEMINI_API_KEY,
)
embeddings_model

In [ ]:
### Read all the PDF with the directory
def process_pdfs(pdf_directory):
    """Process all PDFs in within a Directory"""

    all_docs = []
    pdf_dir = Path(pdf_directory).resolve()
    pdf_files = list(pdf_dir.glob("**/*.pdf"))
    print(f"Found PDF files: {len(pdf_files)}")
    for pdf_file in pdf_files:
        # Process each PDF file
        print(f"Processing {pdf_file.name}")
        try:
            loader = PyMuPDFLoader(str(pdf_file))
            documents = loader.load()
            docs = loader.load()
            for doc in documents:
                doc.metadata["source_file"] = str(pdf_file.name)
                doc.metadata["file_type"] = str(pdf_file.suffix)
            all_docs.extend(docs)
            print(f"FIle Loaded with {len(documents)} documents {pdf_file.name}")

        except Exception as e:
            print(f"Error processing {pdf_file.name}: {e}")

    print(f"Total documents loaded: {len(all_docs)}")
    return all_docs


all_pdf_docs = process_pdfs("data")
# all_pdf_docs

In [ ]:
### Text Splitting
def split_documents(documents, chunk_size=550, chunk_overlap=80):
    """Split documents into smaller chunks"""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""],
    )
    split_docs = text_splitter.split_documents(documents)
    print(f" {len(documents)} documnets is Split into {len(split_docs)} chunks")
    # return split_docs
    if split_docs:
        print("\nExample chunks:")
        print(
            f"Content: {split_docs[0].page_content[:100]}..."
        )  # Print first 100 characters
        print(f"Metadata: {split_docs[0].metadata}")
    return split_docs


#         for i, doc in enumerate(split_docs[:3]):  # Print first 3 chunks
#             print(f"Chunk {i+1}:\n{doc.page_content}\n")

split_chunks = split_documents(all_pdf_docs)

## Embeddings

An **embedding** is a numerical representation (vector) of text that captures its **meaning**. Instead of storing words as plain text, an embedding model converts them into numbers that a computer can compare.

This allows AI systems to search by **meaning (semantic similarity)** instead of exact keywords.

---

### Example

Text:

```
I love dogs.
```

Embedding:

```text
[0.23, -0.45, 0.78, 0.12, ...]
```

Another text:

```
I like puppies.
```

Embedding:

```text
[0.21, -0.43, 0.80, 0.10, ...]
```

Although the words are different, the embeddings are very similar because both sentences have nearly the same meaning.

---

### How Embeddings Are Used in RAG

```text
Document
     │
     ▼
Embedding Model
     │
     ▼
Embedding Vector
     │
     ▼
Vector Database
```

When a user asks a question:

```text
User Question
      │
      ▼
Generate Query Embedding
      │
      ▼
Similarity Search
      │
      ▼
Retrieve Relevant Documents
```

---

### Popular Embedding Models

- OpenAI `text-embedding-3-small`
- OpenAI `text-embedding-3-large`
- Google Gemini Embeddings
- BGE
- E5
- Nomic Embed
- Qwen Embeddings

---

### Key Takeaway

**Embeddings convert text into vectors so computers can understand and compare the meaning of text, making semantic search possible in RAG systems.**


In [ ]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
from time import sleep

from langchain_community import embeddings


class EmbeddingManager:
    def __init__(self, model_name: str = EMBEDDING_MODEL):
        """
        Initialize the EmbeddingManager with a specified model name.
        """
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        """
        Load Google Model.
        """
        try:
            print("Loading Google EEmbedding model...")
            self.model = embeddings_model
            # Generate one sample embedding to check everything is working
            sample = self.model.embed_query("Hello, my dog is cute")
            print(f"Model Loaded Successfully: {self.model_name}")
            print(f"Embedding Dimension: {len(sample)}")
        except Exception as e:
            print(f"Error occurred while loading the model: {e}")
            raise

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """
        Generate embeddings for a list of texts.

        Args:
            texts (List[str]): A list of text strings to embed.

        Returns:
            np.ndarray: An array of embeddings for the input texts.
        """
        if not self.model:
            raise ValueError("Embedding model is not loaded.")
        print(f"Generating embeddings for {len(texts)} texts...")

        embeddings = []

        for i, text in enumerate(texts, start=1):
            print(f"Generating embedding {i}/{len(texts)}")

            embedding = self.model.embed_query(text)
            embeddings.append(embedding)

            # Wait 1 seconds before the next request
            if i < len(texts):
                print("Waiting 1 seconds...")
                sleep(0.8)

            print("Embeddings generated successfully.")

        return np.array(embeddings)

    def get_embedding_dimension(self) -> int:
        """Get the dimensionality of the embeddings."""
        if not self.model:
            raise ValueError("Embedding model is not loaded.")
        return self.model.get_embedding_dimension()


embedding_manager = EmbeddingManager()
embedding_manager

# Vector Store

## Overview

A **Vector Store** (also called a **Vector Database**) is a storage system that stores **embeddings (vectors)** instead of plain text. It allows applications to perform **semantic search**, meaning it retrieves information based on the **meaning** of the text rather than exact keywords.

Vector stores are a core component of **Retrieval-Augmented Generation (RAG)** systems.

---

## Why Do We Use a Vector Store?

Traditional keyword search has limitations:

- Requires exact keyword matches.
- Doesn't understand context or synonyms.
- Becomes inefficient with large datasets.

A vector store solves these problems by:

- Understanding the meaning of text.
- Finding semantically similar documents.
- Retrieving relevant context quickly.
- Improving LLM responses with accurate information.

---

## How Does It Work?

The process is simple:

```text
Documents
    │
    ▼
Split into Chunks
    │
    ▼
Generate Embeddings
    │
    ▼
Store in Vector Store
    │
    ▼
User Query
    │
    ▼
Generate Query Embedding
    │
    ▼
Similarity Search
    │
    ▼
Top Relevant Chunks
    │
    ▼
LLM Generates Final Answer
```

---

## Example

Suppose your knowledge base contains:

```text
Document 1:
LangGraph is used to build AI agents.

Document 2:
PostgreSQL is a relational database.

Document 3:
RAG combines retrieval with LLMs.
```

User asks:

```text
How can I build an AI agent?
```

Instead of searching for the exact words, the vector store understands the meaning of the query and returns:

```text
✅ Document 1:
LangGraph is used to build AI agents.
```

The retrieved document is then provided to the LLM, which uses it to generate a more accurate response.

---

## Popular Vector Stores

- FAISS
- ChromaDB
- PGVector
- Qdrant
- Milvus
- Weaviate
- Pinecone

---

## Key Takeaways

- Stores **embeddings (vectors)** instead of plain text.
- Enables **semantic similarity search**.
- Retrieves the most relevant documents for a query.
- Widely used in **RAG**, AI assistants, chatbots, and recommendation systems.
- Helps LLMs generate more accurate and context-aware responses.


In [ ]:
class VectorStore:
    """Manage all vector-related operations. and Store all Documents with embeddings within ChromaDB"""

    def __init__(
        self,
        collection_name: str = "pdf_documents",
        persist_directory: str = "./data/vectordb_store",
    ):
        """Initialize the vector store.

        Args:
            collection_name (str): The name of the collection in ChromaDB.
            persist_directory (str): The directory where the vector store will persist data.
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        """Initialize the vector store."""
        try:
            # Create DB
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)

            ##Get or create Collection
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={
                    "description": "Collection for storing PDF document embeddings",
                    "version": "1.0",
                },
            )
            print(f"Vector store initialized with collection: {self.collection_name}")
            print(f"Collection metadata: {self.collection.metadata}")
            print(f"Example documents in collection: {self.collection.count()}")
        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise

    def add_document(self, documents: List[Any], embeddings: np.ndarray):
        """Add a document to the vector store.

        Args:
            document (dict): The document to add, with keys "content", "metadata", and "id".
            embedding (list): The embedding for the document.

        Returns:
            None
        """
        if len(documents) != len(embeddings):
            raise ValueError("The number of documents and embeddings must match.")
        print(f"Adding {len(documents)} documents to vector store...")

        ids = []
        metadatas = []
        documents_text = []
        embedding_list = []
        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            source = doc.metadata.get("source", "unknown")
            page = doc.metadata.get("page", 0)

            doc_id = hashlib.sha256(
                f"{source}_{page}_{i}_{doc.page_content}".encode("utf-8")
            ).hexdigest()
            ids.append(doc_id)
            # Crate Metadata
            metadata = doc.metadata.copy()  # copy is recommended
            metadata["doc_index"] = i
            metadata["content_length"] = len(doc.page_content)
            metadatas.append(metadata)

            documents_text.append(doc.page_content)
            embedding_list.append(embedding.tolist())
        try:
            self.collection.add(
                documents=documents_text,
                embeddings=embedding_list,
                metadatas=metadatas,
                ids=ids,
            )
            print(f"Successfully added {len(ids)} documents to the vector store.")
            print(f"Collection now contains {self.collection.count()} documents.")

            print("\nExample Document")
            print("-" * 40)
            print(f"ID      : {ids[0]}")
            print(f"Source  : {documents[0].metadata.get('source')}")
            print(f"Page    : {documents[0].metadata.get('page')}")
            print(f"Content : {documents[0].page_content[:100]}...")
        except Exception as e:
            print(f"Error adding document to vector store: {e}")
            raise


vector_store = VectorStore()
vector_store

In [ ]:
split_chunks

In [ ]:
### Convert the text to embedding ###

texts = [doc.page_content for doc in split_chunks]
texts

In [10]:
##Generate the embeddings ##

embeddings = embedding_manager.generate_embeddings(texts)

Embeddings generated successfully.
Generating embedding 19/151
Waiting 1 seconds...
Embeddings generated successfully.
Generating embedding 20/151
Waiting 1 seconds...
Embeddings generated successfully.
Generating embedding 21/151
Waiting 1 seconds...
Embeddings generated successfully.
Generating embedding 22/151
Waiting 1 seconds...
Embeddings generated successfully.
Generating embedding 23/151
Waiting 1 seconds...
Embeddings generated successfully.
Generating embedding 24/151
Waiting 1 seconds...
Embeddings generated successfully.
Generating embedding 25/151
Waiting 1 seconds...
Embeddings generated successfully.
Generating embedding 26/151
Waiting 1 seconds...
Embeddings generated successfully.
Generating embedding 27/151
Waiting 1 seconds...
Embeddings generated successfully.
Generating embedding 28/151
Waiting 1 seconds...
Embeddings generated successfully.
Generating embedding 29/151
Waiting 1 seconds...
Embeddings generated successfully.
Generating embedding 30/151
Waiting 1 sec

In [11]:
##Store in Database
vector_store.add_document(split_chunks, embeddings)

Adding 151 documents to vector store...
Successfully added 151 documents to the vector store.
Collection now contains 151 documents.

Example Document
----------------------------------------
ID      : 36d576058a23035dd7b1281c3a2ef5af49ed5e111da0ecb1917e43f3c5f45020
Source  : C:\Users\kisho\WorkSpace\learning\ai workflow\langchain\RAG\data\pdf\AI Engineer Roadmap 2026.pdf
Page    : 0
Content : Transform a production-grade full stack engineer
(React/Next.js/Node.js/TypeScript/PostgreSQL/Redis/...


# Retrieval in RAG (Retrieval-Augmented Generation)

## Overview

**Retrieval** is the process of finding the most relevant information from a knowledge base before sending a user's query to a Large Language Model (LLM). Instead of relying only on the model's trained knowledge, retrieval provides external context, enabling the LLM to generate more accurate and up-to-date responses.

---

## What Problem Does Retrieval Solve?

LLMs have several limitations:

- Knowledge is limited to their training data.
- Cannot answer questions about private or newly added documents.
- May generate incorrect or hallucinated responses.
- Fine-tuning for every new document is expensive.

Retrieval solves these problems by fetching relevant information from an external knowledge base at query time.

---

## How Does Retrieval Work?

The retrieval process consists of the following steps:

```text
User Question
      │
      ▼
Generate Query Embedding
      │
      ▼
Search Vector Store
      │
      ▼
Retrieve Top-K Relevant Chunks
      │
      ▼
Send Chunks + Question to LLM
      │
      ▼
Generate Final Answer
```

The vector store compares the query embedding with stored document embeddings using similarity search (e.g., Cosine Similarity) and returns the most relevant document chunks.

---

## Example

Suppose your knowledge base contains:

```text
Document 1:
LangGraph is a framework for building stateful AI agents.

Document 2:
PostgreSQL is a relational database.

Document 3:
RAG combines retrieval with LLM generation.
```

User asks:

> **"What framework can I use to build AI agents?"**

The retrieval system searches the vector database and finds the most relevant chunk:

```text
LangGraph is a framework for building stateful AI agents.
```

This retrieved context is passed to the LLM, which generates a more accurate answer.

---

## Advantages

- Provides accurate and relevant context.
- Reduces hallucinations.
- Supports private and domain-specific knowledge.
- Works with continuously updated documents.
- No need to retrain the LLM when documents change.

---

## Limitations

- Retrieval quality depends on the embedding model.
- Poor chunking can reduce search accuracy.
- Incorrect or missing documents lead to poor responses.
- Searching very large datasets may require optimized vector databases.

---

## Summary

Retrieval is the core of a RAG system. It connects user queries with the most relevant information stored in a vector database, allowing the LLM to answer using real, up-to-date, and domain-specific knowledge instead of relying solely on its pre-trained memory.


In [22]:
from pprint import pprint


class RAGRetriever:
    """Handles query based retrieval of relevant documents. from the vector store."""

    def __init__(self, vector_store, embedding_manager):
        """Initialize the RAGRetriever.

        Args:
            vector_store: The vector store to retrieve documents from.
            embedding_manager: The embedding manager for generating embeddings.
        """
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(
        self, query: str, top_k: int = 10, score_threshold: float = 0.5
    ) -> List[Dict[str, Any]]:
        """
        Retrieve relevant documents based on a query.

        Args:
            query (str): The query string.
            top_k (int): The number of top documents to retrieve.
            score_threshold (float): The minimum similarity score for a document to be considered relevant.

        Returns:
            List[Document]: A list of retrieved documents.
        """
        print(f"Retrieving documents for query: {query}")
        print(f"Score threshold: {score_threshold} and Top-k: {top_k}")

        # Generate embedding for the query
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]

        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()], n_results=top_k
            )
            # pprint(results)
            retrieved_docs = []
            if results["documents"] and results["documents"][0]:
                documents = results["documents"][0]
                metadatas = results["metadatas"][0]
                distances = results["distances"][0]
                ids = results["ids"][0]
                for i, (doc_id, document, metadata, distance) in enumerate(
                    zip(ids, documents, metadatas, distances)
                ):

                    similarity_score = 1 - distance

                    print(
                        f"Rank {i+1}: distance={distance:.4f}, similarity={similarity_score:.10f}"
                    )
                    if similarity_score >= score_threshold:
                        retrieved_docs.append(
                            {
                                "id": doc_id,
                                "content": document,
                                "metadata": metadata,
                                "similarity_score": similarity_score,
                                "distance": distance,
                                "rank": i + 1,
                            }
                        )
                print(f"Retrieved {len(retrieved_docs)} documents. After filtering")
            else:
                print("No documents retrieved.")
            return retrieved_docs

        except Exception as e:
            print(f"Error adding document to vector store: {e}")
            return []


rag_retriever = RAGRetriever(vector_store, embedding_manager)

In [23]:
rag_retriever

In [24]:
rag_retriever.retrieve("DOCUMENT 3 – DEVELOPMENT INVESTMENT")

Retrieving documents for query: DOCUMENT 3 – DEVELOPMENT INVESTMENT
Score threshold: 0.5 and Top-k: 10
Generating embeddings for 1 texts...
Generating embedding 1/1
Embeddings generated successfully.
Rank 1: distance=0.6493, similarity=0.3507198095
Rank 2: distance=0.6942, similarity=0.3058476448
Rank 3: distance=0.6951, similarity=0.3049211502
Rank 4: distance=0.6982, similarity=0.3018007874
Rank 5: distance=0.7174, similarity=0.2826185226
Rank 6: distance=0.7341, similarity=0.2658576965
Rank 7: distance=0.7396, similarity=0.2604174018
Rank 8: distance=0.7481, similarity=0.2518625259
Rank 9: distance=0.7523, similarity=0.2477130890
Rank 10: distance=0.7682, similarity=0.2318025827
Retrieved 0 documents. After filtering


[]

# Vector Database Integration with LLM

## Overview

A **Vector Database** (such as ChromaDB, FAISS, PGVector, or Qdrant) stores document **embeddings (vectors)** and enables an LLM to retrieve relevant information before generating a response.

Instead of relying only on its pre-trained knowledge, the LLM uses the vector database as an external knowledge source. This approach is called **Retrieval-Augmented Generation (RAG)**.

---

## Why Do We Need a Vector Database?

Large Language Models have some limitations:

- Knowledge is limited to training data.
- Cannot access private company documents.
- Cannot answer questions about newly added information.
- May generate hallucinated or incorrect responses.

A vector database solves these problems by providing the most relevant documents at query time.

---

## How Does It Work?

```text
Documents
    │
    ▼
Split into Chunks
    │
    ▼
Generate Embeddings
    │
    ▼
Store in Vector Database
    │
──────────────────────────────────
           User Query
                │
                ▼
     Generate Query Embedding
                │
                ▼
      Search Vector Database
                │
                ▼
      Retrieve Top-K Chunks
                │
                ▼
Question + Retrieved Context
                │
                ▼
              LLM
                │
                ▼
          Final Answer
```

---

## Example

Suppose your vector database contains:

```text
Document 1:
LangGraph is used to build stateful AI agents.

Document 2:
ChromaDB is a vector database used in RAG.

Document 3:
PostgreSQL is a relational database.
```

A user asks:

> **"Which database is used for storing embeddings?"**

### Step 1: Query Embedding

The user's question is converted into an embedding.

```text
"Which database is used for storing embeddings?"

↓

[0.21, 0.45, ..., 384 values]
```

### Step 2: Vector Search

The vector database compares the query embedding with stored document embeddings.

Similarity results:

```text
Document 2 → 0.98 ✅

Document 1 → 0.72

Document 3 → 0.40
```

### Step 3: Retrieve Context

The most relevant document is returned:

```text
ChromaDB is a vector database used in RAG.
```

### Step 4: Send to LLM

The prompt becomes:

```text
Question:
Which database is used for storing embeddings?

Context:
ChromaDB is a vector database used in RAG.
```

The LLM now generates an accurate answer based on the retrieved context.

---

## Why Is This Better?

Without a Vector Database:

```text
Question
   │
   ▼
  LLM
   │
   ▼
May hallucinate or use outdated knowledge
```

With a Vector Database:

```text
Question
   │
   ▼
Vector Database
   │
   ▼
Relevant Documents
   │
   ▼
LLM
   │
   ▼
Accurate, context-aware answer
```

---

## Real-Life Example

Imagine you're building an AI assistant for your company.

The company has:

- Employee handbook
- HR policies
- Product documentation
- Internal APIs

A user asks:

> **"How many leave days do employees get?"**

The LLM doesn't know your company's policy.

Instead:

1. The query is converted into an embedding.
2. The vector database retrieves the HR policy document.
3. The retrieved policy is sent to the LLM.
4. The LLM answers using the actual company document.

---

## Summary

A Vector Database acts as the **knowledge retrieval layer** between your documents and the LLM.

It:

- Stores document embeddings.
- Performs semantic similarity search.
- Retrieves the most relevant information.
- Sends the retrieved context to the LLM.
- Enables accurate, up-to-date, and domain-specific responses without retraining the model.


In [25]:
import os
from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

load_dotenv()
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")

model = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0.2,
    max_retries=2,
    max_tokens=2024,
    # other params...
)

In [31]:
##Simple RAG
def rag_simple_query(query, retriever, model, top_k=10):
    # Implementation for simple RAG query
    results = retriever.retrieve(query, top_k=top_k)
    context: str = "\n\n".join([doc["content"] for doc in results]) if results else ""
    if not context:
        return "No relevant information found."
    ## Generate Answer using LLM
    prompt = f"""Use the Following context to answer the context:\n\n{context}\n\nQuestion: {query}\n\nAnswer:"""
    res = model.invoke([prompt.format(context=context, query=query)])
    return res.content
    

In [47]:
answer =  rag_simple_query("What is AI Engineer Roadmap?", rag_retriever, model)
print(answer)

Retrieving documents for query: What is AI Engineer Roadmap?
Score threshold: 0.5 and Top-k: 10
Generating embeddings for 1 texts...
Generating embedding 1/1
Embeddings generated successfully.
Rank 1: distance=0.4224, similarity=0.5775606930
Rank 2: distance=0.4259, similarity=0.5741084218
Rank 3: distance=0.4454, similarity=0.5545679629
Rank 4: distance=0.4648, similarity=0.5352288783
Rank 5: distance=0.4890, similarity=0.5109964609
Rank 6: distance=0.5062, similarity=0.4938204288
Rank 7: distance=0.5063, similarity=0.4937373996
Rank 8: distance=0.5134, similarity=0.4866080880
Rank 9: distance=0.5346, similarity=0.4654164314
Rank 10: distance=0.5401, similarity=0.4599092603
Retrieved 5 documents. After filtering
The AI Engineer Roadmap is a 6-month plan (approximately 20 hours/week) to transform a production-grade full stack engineer into a production AI engineer. The roadmap consists of the following steps:

1. Introduction
2. Complete Roadmap Overview
3. 6 Month Timeline
4. Weekly L